# Workout Pose Classifier v3 (5 kelas, 3 sumber data, dipandu `dataset_index.csv`)

**Pipeline:** video → MediaPipe 33 keypoint → window 30 frame → 8 sudut sendi → ringkasan per window → Random Forest

Yang berubah dari v2:
- Daftar video, label kelas, label kualitas, split train/test, dan status **dibaca dari `dataset_index.csv`**. Nama file video dicocokkan lewat kolom `file` di `dataset_index_named.csv` (contoh `00228118.mp4`, `Copy of push up 1.mp4`).
- Video dikelompokkan jadi **3 folder sesuai sumber dataset**: `Qevd`, `SevenFitness` (7 workout / multiview), dan `Pushup`.
- Kelas final hanya 5: `Elbow_Plank`, `High_Plank`, `Pushup`, `Pushup_on_Knees`, `Side_Plank`. Baris CSV dengan kelas lain (termasuk squat) otomatis dilewati dan dilaporkan.
- Mode `append` / `rebuild` dan `dataset_final.npz` lama dihapus. Semua data dibangun ulang dari CSV + video (hasil ekstraksi tetap dicache, jadi cepat di run berikutnya).
- Frame sampling per video mengikuti CSV (`fps_asli` / `fps_efektif`), bukan satu angka `EVERY_N` untuk semua.

Dua model:
1. **kelas**: gerakan apa. Pakai semua sumber.
2. **kualitas**: good / bad. Cuma dari sumber yang punya label kualitas (`multiview` dan `pushup`). QEVD tidak punya label itu.

## Struktur folder (WAJIB persis)

```
Project ML/Dataset/
├── dataset_index_named.csv      # CSV dengan kolom `file` (nama video asli)
├── Videos/                      # dibuat otomatis kalau belum ada (huruf besar/kecil tidak masalah)
│   ├── Qevd/           00228118.mp4, 00228119.mp4, ...
│   ├── SevenFitness/   subject_001_push_up_bad_diagonal.mp4, ...
│   └── Pushup/         Copy of push up 1.mp4, ...
├── keypoints/                   # otomatis: cache .npy per video
└── outputs/                     # otomatis: model + dataset_final_v3.npz
```

Aturan:
- Taruh video ke 3 folder di atas sesuai sumbernya. Nama file harus sama dengan kolom `file` di CSV. Spasi di nama file tidak masalah, huruf besar/kecil diabaikan.
- Di dalam tiap folder boleh ada subfolder (misalnya `Correct sequence`, `Wrong sequence`, `front`, `Part-4`), file dicari rekursif.
- Ekstensi `.mp4 .mov .avi .mkv`.
- Kalau semua video masih tercampur di satu folder, pakai bagian 4 untuk memilah otomatis ke 3 folder.
- Kalau file `keypoints/<id>.npy` sudah ada, videonya tidak perlu ada lagi.
- Kalau folder video kamu ada di tempat lain (mis. OneDrive), tidak perlu dipindah: ubah `SRC_DIRS` di bagian 1.

## 1. Config

In [4]:
import re, shutil, warnings
from pathlib import Path
from collections import Counter

import joblib
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

warnings.filterwarnings("ignore")
pd.set_option("display.width", 140)

# ---- kelas final (urutan = index y_kelas) ----
KELAS = ["Elbow_Plank", "High_Plank", "Pushup", "Pushup_on_Knees", "Side_Plank"]
ALIAS = {"push_up": "Pushup", "push-up": "Pushup"}      # nama alternatif di CSV
SUMBER = ("qevd", "multiview", "pushup")               # = 3 folder dataset
PUNYA_KUALITAS = {"qevd": False, "multiview": True, "pushup": True}

# ---- path ----
IN_COLAB = Path("/content/drive/MyDrive").exists()
ROOT = Path(r"C:\Users\hanse\OneDrive\Documents\MOVIT\Dataset")
CSV_PATH = ROOT / "dataset_index_named.csv"      # CSV yang sudah punya kolom nama file asli (`file`)


def cari_folder(parent, nama):
    """cari subfolder `nama` tanpa peduli huruf besar/kecil (Videos = videos); kalau belum ada pakai nama aslinya"""
    parent = Path(parent)
    if parent.exists():
        for p in parent.iterdir():
            if p.is_dir() and p.name.lower() == nama.lower():
                return p
    return parent / nama


VIDEO_ROOT = cari_folder(ROOT, "Videos")
KP, OUT_DIR = ROOT / "keypoints", ROOT / "outputs"

# 3 folder video, satu per sumber. Kalau video sudah ada di tempat lain dan nggak mau dipindah,
# ganti path-nya di sini (isinya boleh berupa file langsung atau subfolder).
FOLDER_VIDEO = {"qevd": "Qevd", "multiview": "SevenFitness", "pushup": "Pushup"}   # nama 3 folder di disk
SRC_DIRS = {s: cari_folder(VIDEO_ROOT, FOLDER_VIDEO[s]) for s in SUMBER}
# contoh (Windows):
# SRC_DIRS["qevd"] = Path(r"C:\Users\NAMA\OneDrive\Documents\MOVIT\Qevd")

for p in [KP, OUT_DIR, *SRC_DIRS.values()]:
    p.mkdir(parents=True, exist_ok=True)

# ---- pemakaian CSV ----
STATUS_DIPAKAI = {"ok"}      # tambah "review" kalau mau ikut pakai video yang NaN-nya banyak
EVERY_N_DEFAULT = 2          # dipakai hanya kalau fps_asli / fps_efektif di CSV kosong

# ---- window ----
WINDOW = 30        # frame per window
STRIDE = 10        # geser antar window
MAX_MISSING = 0.2  # window dibuang kalau >20% frame tanpa pose

# ---- 8 sudut sendi (nama, titik a, titik sudut b, titik c) indeks MediaPipe ----
SUDUT_DEF = [
    ("siku_kiri", 11, 13, 15), ("siku_kanan", 12, 14, 16),
    ("bahu_kiri", 23, 11, 13), ("bahu_kanan", 24, 12, 14),
    ("pinggul_kiri", 11, 23, 25), ("pinggul_kanan", 12, 24, 26),
    ("lutut_kiri", 23, 25, 27), ("lutut_kanan", 24, 26, 28),
]
NAMA_SUDUT = [s[0] for s in SUDUT_DEF]
print("root:", ROOT)
print("folder video:", {k: str(v) for k, v in SRC_DIRS.items()})

root: C:\Users\hanse\OneDrive\Documents\MOVIT\Dataset
folder video: {'qevd': 'C:\\Users\\hanse\\OneDrive\\Documents\\MOVIT\\Dataset\\Videos\\Qevd', 'multiview': 'C:\\Users\\hanse\\OneDrive\\Documents\\MOVIT\\Dataset\\Videos\\SevenFitness', 'pushup': 'C:\\Users\\hanse\\OneDrive\\Documents\\MOVIT\\Dataset\\Videos\\Pushup'}


## 2. Fungsi keypoint, window, sudut

Video diproses sekali, hasilnya dicache di `keypoints/<id>.npy`. Run ulang tidak mengulang ekstraksi.
Sudut dihitung dari keypoint yang sama untuk semua sumber, jadi konsisten.

In [5]:
def norm_kelas(name):
    k = str(name).lower().strip().replace(" ", "_")
    for kn in KELAS:
        if kn.lower() == k:
            return kn
    return ALIAS.get(k)


def video_to_keypoints(path, step):
    """video -> (T, 33, 2) normalised xy; frame tanpa pose = NaN. Ambil tiap frame ke-`step`."""
    import cv2, mediapipe as mp          # diimport di sini supaya cache .npy tetap jalan tanpa mediapipe
    frames, i = [], 0
    cap = cv2.VideoCapture(str(path))
    with mp.solutions.pose.Pose(min_detection_confidence=0.5) as pose:
        while cap.isOpened():
            ok, frame = cap.read()
            if not ok:
                break
            i += 1
            if i % step:
                continue
            res = pose.process(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
            if res.pose_landmarks:
                frames.append([[l.x, l.y] for l in res.pose_landmarks.landmark])
            else:
                frames.append(np.full((33, 2), np.nan))
    cap.release()
    return np.array(frames, dtype=np.float32).reshape(-1, 33, 2)


def get_keypoints(path, video_id, step):
    cache = KP / f"{video_id}.npy"
    if cache.exists():
        return np.load(cache)[:, :33, :2].astype(np.float32)
    kp = video_to_keypoints(path, step)
    np.save(cache, kp)
    return kp


def make_windows(kp):
    out = []
    for s in range(0, len(kp) - WINDOW + 1, STRIDE):
        w = kp[s:s + WINDOW]
        miss = np.isnan(w).any(axis=(1, 2))
        if miss.mean() > MAX_MISSING or miss.all():
            continue
        if miss.any():   # isi frame kosong dengan interpolasi
            w = (pd.DataFrame(w.reshape(WINDOW, -1)).interpolate(limit_direction="both")
                 .to_numpy().reshape(w.shape))
        out.append((s, w))
    return out


def angles(X):
    """(N, T, 33, 2) -> (N, T, 8) sudut dalam derajat"""
    out = []
    for _, a, b, c in SUDUT_DEF:
        ba, bc = X[:, :, a] - X[:, :, b], X[:, :, c] - X[:, :, b]
        cos = (ba * bc).sum(-1) / (np.linalg.norm(ba, axis=-1) * np.linalg.norm(bc, axis=-1) + 1e-6)
        out.append(np.degrees(np.arccos(np.clip(cos, -1, 1))))
    return np.stack(out, -1).astype(np.float32)


def summarize(A):
    """(N, T, 8) -> (N, 40): min, max, mean, std, range tiap sudut"""
    return np.concatenate([A.min(1), A.max(1), A.mean(1), A.std(1), A.max(1) - A.min(1)], axis=1)


FEAT_COLS = [f"{n}_{s}" for s in ("min", "max", "mean", "std", "range") for n in NAMA_SUDUT]

## 3. Baca `dataset_index.csv`

Yang dipakai: baris dengan kelas di `KELAS`, sumber di `SUMBER`, dan `status` di `STATUS_DIPAKAI`.
Selain itu dilewati dan dilaporkan di bawah. Kolom `split` di CSV dipakai apa adanya.

In [6]:
NAMA_KOLOM = ("file", "nama_file", "filename", "video", "video_name")   # kolom nama file asli (opsional)
idx = pd.read_csv(CSV_PATH, dtype={c: str for c in NAMA_KOLOM})
butuh = {"id", "sumber", "kelas", "kualitas", "split", "status"}
assert "file" in idx.columns, "CSV ini belum punya kolom `file`; pakai dataset_index_named.csv"
assert butuh <= set(idx.columns), f"kolom CSV kurang: {butuh - set(idx.columns)}"
print("baris di CSV:", len(idx))

idx["sumber"] = idx["sumber"].astype(str).str.lower().str.strip()
idx["kelas_n"] = idx["kelas"].map(norm_kelas)
idx["kualitas"] = idx["kualitas"].fillna("unknown").astype(str).str.lower()
idx["view"] = idx["view"].fillna("") if "view" in idx.columns else ""

# --- yang dilewati ---
alasan = Counter()
for _, r in idx[idx.kelas_n.isna()].iterrows():
    alasan[f"kelas '{r.kelas}' bukan 5 kelas final"] += 1
for _, r in idx[~idx.sumber.isin(SUMBER)].iterrows():
    alasan[f"sumber '{r.sumber}' tidak dikenal"] += 1
for _, r in idx[~idx.status.isin(STATUS_DIPAKAI)].iterrows():
    alasan[f"status '{r.status}' tidak dipakai"] += 1
keep = idx.kelas_n.notna() & idx.sumber.isin(SUMBER) & idx.status.isin(STATUS_DIPAKAI)
if alasan:
    print("\ndilewati:")
    for k, v in alasan.items():
        print(f"  {v:4d}  {k}")
idx = idx[keep].copy()

# kualitas hanya berlaku untuk sumber yang punya label kualitas
idx.loc[~idx.sumber.map(PUNYA_KUALITAS), "kualitas"] = "unknown"
bad_q = idx[idx.sumber.map(PUNYA_KUALITAS) & ~idx.kualitas.isin(["good", "bad"])]
assert bad_q.empty, f"{len(bad_q)} baris multiview/pushup tanpa label good/bad, contoh: {bad_q.id.head(3).tolist()}"
assert idx.split.isin(["train", "test"]).all(), "kolom split harus train / test"
assert idx.id.is_unique, "kolom id harus unik"

# nama file asli video: dari kolom nama_file / file / filename (kalau ada), kalau tidak ada pakai id
kol_nama = next((c for c in NAMA_KOLOM if c in idx.columns), None)
if kol_nama:
    idx["nama_vid"] = idx[kol_nama].fillna("").astype(str).map(lambda x: Path(x).stem.strip().lower())
    print(f"nama file video dibaca dari kolom '{kol_nama}' ({(idx.nama_vid != '').sum()} terisi)")
else:
    idx["nama_vid"] = ""
    print("CSV tidak punya kolom nama file asli, nama file dianggap sama dengan kolom id")

# group = satu orang tidak boleh pecah ke train dan test (multiview punya kolom subjek)
if "subjek" in idx.columns:
    subj = idx["subjek"]
else:
    subj = pd.Series(np.nan, index=idx.index)
idx["group"] = np.where((idx.sumber == "multiview") & subj.notna(),
                        "multiview_s" + subj.fillna(0).astype(int).astype(str), idx.id)

# langkah sampling frame per video: fps_asli / fps_efektif
if {"fps_asli", "fps_efektif"} <= set(idx.columns):
    step = (idx.fps_asli / idx.fps_efektif).round()
    idx["step"] = step.where(step >= 1).fillna(EVERY_N_DEFAULT).astype(int)
else:
    idx["step"] = EVERY_N_DEFAULT

print(f"\nvideo dipakai: {len(idx)}")
display(pd.crosstab([idx.sumber, idx.kelas_n], idx.kualitas, margins=True))
display(pd.crosstab(idx.sumber, idx.split))
display(idx.groupby("sumber").step.value_counts().rename("jumlah video").to_frame())

baris di CSV: 1744

dilewati:
    73  status 'review' tidak dipakai
nama file video dibaca dari kolom 'file' (1671 terisi)

video dipakai: 1671


kualitas                   bad  good  unknown   All
sumber    kelas_n                                  
multiview Pushup            72    71        0   143
pushup    Pushup            50    50        0   100
qevd      Elbow_Plank        0     0      287   287
          High_Plank         0     0      283   283
          Pushup             0     0      270   270
          Pushup_on_Knees    0     0      289   289
          Side_Plank         0     0      299   299
All                        122   121     1428  1671

split,test,train
sumber,,
multiview,30,113
pushup,20,80
qevd,216,1212


,,jumlah video
sumber,step,
multiview,1,143
pushup,1,100
qevd,2,1428


## 4. (Opsional) Pilah video ke 3 folder sumber

Kalau semua video masih ada di satu folder campur, isi `DUMP_DIR` lalu run cell ini. Video dicocokkan lewat nama file di CSV, lalu disalin ke `Videos/Qevd`, `Videos/SevenFitness`, atau `Videos/Pushup`.
Kalau video sudah ada di 3 folder, biarkan `DUMP_DIR = None`.

In [7]:
DUMP_DIR = None      # contoh: ROOT / "semua_video"
PINDAH = False       # False = salin (aman), True = pindahkan

VIDEO_EXT = {".mp4", ".mov", ".avi", ".mkv"}

if DUMP_DIR is not None:
    peta = {}
    for r in idx.itertuples():
        peta[r.id.lower()] = r.sumber
        if r.nama_vid:
            peta[r.nama_vid] = r.sumber
    n = Counter()
    for p in tqdm(sorted(Path(DUMP_DIR).rglob("*")), desc="memilah"):
        if p.suffix.lower() not in VIDEO_EXT or p.stem.lower() not in peta:
            continue
        tujuan = SRC_DIRS[peta[p.stem.lower()]] / p.name
        if not tujuan.exists():
            (shutil.move if PINDAH else shutil.copy2)(p, tujuan)
        n[peta[p.stem.lower()]] += 1
    print("dipilah per sumber:", dict(n))
else:
    print("DUMP_DIR kosong, bagian ini dilewati")

DUMP_DIR kosong, bagian ini dilewati


## 5. Cocokkan CSV dengan file video

Cek dulu apa yang ketemu sebelum ekstraksi. Baris yang videonya tidak ada **dan** tidak punya cache keypoint akan dilaporkan dan dilewati.

In [8]:
def index_folder(d):
    out = {}
    if Path(d).exists():
        for p in sorted(Path(d).rglob("*")):
            if p.suffix.lower() in VIDEO_EXT:
                out.setdefault(p.stem.lower(), p)
    return out


def kunci(r):
    # urutan prioritas: nama file asli dari CSV, lalu id persis, lalu nomor id saja
    ks = [r.nama_vid] if r.nama_vid else []
    return ks + [r.id.lower(), r.id.split("_", 1)[-1].lower()]


folder_idx = {s: index_folder(SRC_DIRS[s]) for s in SUMBER}
paths, ada_cache, hilang = [], [], {s: [] for s in SUMBER}
for r in idx.itertuples():
    p = next((folder_idx[r.sumber][k] for k in kunci(r) if k in folder_idx[r.sumber]), None)
    c = (KP / f"{r.id}.npy").exists()
    paths.append(p); ada_cache.append(c)
    if p is None and not c:
        hilang[r.sumber].append(r.nama_vid or r.id)
idx["path"] = paths
idx["ada_cache"] = ada_cache

ITEMS = idx[idx.path.notna() | idx.ada_cache].copy()
print(f"siap diproses: {len(ITEMS)} dari {len(idx)} video")
display(pd.crosstab(ITEMS.sumber, ITEMS.split, margins=True))
for s, v in hilang.items():
    if v:
        print(f"[{s}] {len(v)} video di CSV tidak ketemu di {SRC_DIRS[s]} (contoh nama: {v[:3]})")
assert len(ITEMS), "tidak ada video yang ketemu. Cek nama file video cocok dengan kolom `file` di CSV dan isi 3 folder videos/"

siap diproses: 1671 dari 1671 video


split,test,train,All
sumber,,,
multiview,30,113,143
pushup,20,80,100
qevd,216,1212,1428
All,266,1405,1671


## 6. Ekstraksi video → keypoint → window

Bagian paling lama (sekitar 1,5 jam dengan 1 worker; dengan `N_WORKERS = 3` biasanya jauh lebih singkat). Dibagi dua langkah supaya **tahan crash**:

- **6a** menjalankan MediaPipe di beberapa *proses terpisah* (paralel, diatur `N_WORKERS`). Kalau satu video bikin OpenCV/MediaPipe crash, yang mati cuma proses itu, bukan notebook. Video penyebabnya dicatat di `keypoints/_gagal.txt`, dilewati, dan ekstraksi lanjut otomatis.
- Hasil tiap video langsung disimpan di `keypoints/<id>.npy`. Kalau notebook ditutup atau dihentikan, run ulang cell 6a **melanjutkan** dari yang belum selesai.
- **6b** membuat window dari keypoint yang sudah ada dan mengecek jumlah frame terhadap kolom `frames` di CSV.

In [9]:
import json, subprocess, sys

WORKER = ROOT / "_extract_worker.py"
WORKER_CODE = r"""
import sys, json
from pathlib import Path
import numpy as np

def main():
    job = json.load(open(sys.argv[1], encoding="utf-8"))
    kp_dir = Path(job["kp_dir"])
    import cv2, mediapipe as mp
    for it in job["items"]:
        out = kp_dir / (it["id"] + ".npy")
        if out.exists():
            continue
        (kp_dir / job["progress"]).write_text(it["id"], encoding="utf-8")   # penanda: video yang sedang diproses
        frames, i = [], 0
        cap = cv2.VideoCapture(it["path"])
        with mp.solutions.pose.Pose(min_detection_confidence=0.5) as pose:
            while cap.isOpened():
                ok, frame = cap.read()
                if not ok:
                    break
                i += 1
                if i % it["step"]:
                    continue
                res = pose.process(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
                if res.pose_landmarks:
                    frames.append([[l.x, l.y] for l in res.pose_landmarks.landmark])
                else:
                    frames.append(np.full((33, 2), np.nan))
        cap.release()
        np.save(out, np.array(frames, dtype=np.float32).reshape(-1, 33, 2))
        print("OK", it["id"], len(frames), flush=True)

main()
"""
WORKER.write_text(WORKER_CODE, encoding="utf-8")
print("worker:", WORKER)

worker: C:\Users\hanse\OneDrive\Documents\MOVIT\Dataset\_extract_worker.py


In [10]:
from concurrent.futures import ThreadPoolExecutor
from threading import Lock

N_WORKERS = 3     # jumlah proses MediaPipe paralel. Naikkan (4-6) kalau CPU masih santai, turunkan kalau laptop lag/panas

GAGAL_FILE = KP / "_gagal.txt"
gagal = {x for x in GAGAL_FILE.read_text(encoding="utf-8").split("\n") if x} if GAGAL_FILE.exists() else set()
# hapus _gagal.txt kalau mau mencoba ulang video yang dulu gagal
lock = Lock()


def belum(items):
    return [r for r in items
            if r.id not in gagal and not (KP / f"{r.id}.npy").exists() and r.path is not None]


todo0 = belum(list(ITEMS.itertuples()))
print(f"sudah ada cache: {len(ITEMS) - len(todo0) - len(gagal)} | perlu diekstrak: {len(todo0)} | "
      f"dilewati (gagal sebelumnya): {len(gagal)} | worker paralel: {N_WORKERS}")
bar = tqdm(total=len(todo0), desc="ekstraksi")


def jalur(k, items):
    # satu jalur worker: jalankan proses, kalau crash catat videonya, lewati, lanjutkan sisanya
    prog, logf = KP / f"_sedang_{k}.txt", KP / f"_worker_{k}.log"
    for _ in range(len(items) + 5):
        todo = belum(items)
        if not todo:
            return
        job = KP / f"_job_{k}.json"
        job.write_text(json.dumps(dict(kp_dir=str(KP), progress=prog.name,
                                       items=[dict(id=r.id, path=str(r.path), step=int(r.step)) for r in todo])),
                       encoding="utf-8")
        prog.unlink(missing_ok=True)
        with open(logf, "a", encoding="utf-8") as log:
            p = subprocess.Popen([sys.executable, str(WORKER), str(job)], stdout=subprocess.PIPE, stderr=log, text=True)
            for line in p.stdout:
                if line.startswith("OK"):
                    bar.update(1)
            p.wait()
        if p.returncode == 0:
            continue
        sedang = prog.read_text(encoding="utf-8").strip() if prog.exists() else ""
        if not sedang:
            tail = logf.read_text(encoding="utf-8", errors="ignore").splitlines()[-15:]
            raise RuntimeError("proses ekstraksi berhenti sebelum video pertama (mungkin mediapipe/opencv belum terpasang):\n"
                               + "\n".join(tail))
        with lock:
            nama = idx.loc[idx.id == sedang, "nama_vid"].iloc[0] or sedang
            print(f"\n[worker {k}] crash di video {sedang} ({nama}), kode {p.returncode}. Dilewati, lanjut.")
            gagal.add(sedang)
            GAGAL_FILE.write_text("\n".join(sorted(gagal)), encoding="utf-8")
    raise RuntimeError(f"worker {k}: terlalu banyak percobaan ulang, cek {logf}")


chunks = [todo0[k::N_WORKERS] for k in range(N_WORKERS)]      # dibagi selang-seling supaya beban merata
with ThreadPoolExecutor(max_workers=N_WORKERS) as ex:
    for f in [ex.submit(jalur, k, c) for k, c in enumerate(chunks) if c]:
        f.result()                                              # kalau ada error di worker, muncul di sini
bar.close()

ok = sum((KP / f"{r.id}.npy").exists() for r in ITEMS.itertuples())
print(f"\nselesai. keypoint tersedia: {ok}/{len(ITEMS)} video | gagal/dilewati: {len(gagal)}")
if gagal:
    print("video yang dilewati (cek apakah file-nya rusak):")
    for g in sorted(gagal):
        print("  ", g, "->", idx.loc[idx.id == g, "nama_vid"].iloc[0])

sudah ada cache: 224 | perlu diekstrak: 1447 | dilewati (gagal sebelumnya): 0 | worker paralel: 3


ekstraksi:   0%|          | 0/1447 [00:00<?, ?it/s]


selesai. keypoint tersedia: 1671/1671 video | gagal/dilewati: 0


In [11]:
Xs, rows, kosong, beda_frame = [], [], [], []
for r in tqdm(list(ITEMS.itertuples()), desc="window"):
    cache = KP / f"{r.id}.npy"
    if r.id in gagal or not cache.exists():
        continue
    kp = np.load(cache)[:, :33, :2].astype(np.float32)
    if hasattr(r, "frames") and pd.notna(r.frames) and abs(len(kp) - int(r.frames)) > 2:
        beda_frame.append((r.id, len(kp), int(r.frames)))
    ws = make_windows(kp)
    if not ws:
        kosong.append(r.id)
    for s, w in ws:
        Xs.append(w)
        rows.append(dict(video_id=r.id, mulai_frame=s, sumber=r.sumber, kelas=r.kelas_n,
                         kualitas=r.kualitas, view=r.view, split=r.split, group=r.group))

if kosong:
    print(f"{len(kosong)} video tidak menghasilkan window (terlalu pendek / pose hilang), contoh:", kosong[:3])
if beda_frame:
    print(f"{len(beda_frame)} video jumlah frame beda dengan CSV (id, hasil, csv), contoh:", beda_frame[:3])
    print("  kalau banyak, cek fps video asli / kolom step.")
assert Xs, "tidak ada window sama sekali"

X_all = np.stack(Xs).astype(np.float32)
META = pd.DataFrame(rows)
A_all = angles(X_all)
assert len(META) == len(X_all) == len(A_all)
assert META.kelas.isin(KELAS).all()

print("\nTOTAL window:", X_all.shape)
display(pd.crosstab([META.sumber, META.kelas], META.kualitas, margins=True))
display(pd.crosstab(META.sumber, META.split))
print("orang/video yang kepecah ke train & test:", int((META.groupby("group").split.nunique() > 1).sum()), "(harus 0)")

window:   0%|          | 0/1671 [00:00<?, ?it/s]

3 video tidak menghasilkan window (terlalu pendek / pose hilang), contoh: ['multiview_00142', 'qevd_02340', 'qevd_03653']

TOTAL window: (12737, 30, 33, 2)


kualitas                    bad  good  unknown    All
sumber    kelas                                      
multiview Pushup           1969  2069        0   4038
pushup    Pushup            524   328        0    852
qevd      Elbow_Plank         0     0     1427   1427
          High_Plank          0     0     1410   1410
          Pushup              0     0     1435   1435
          Pushup_on_Knees     0     0     1569   1569
          Side_Plank          0     0     2006   2006
All                        2493  2397     7847  12737

split,test,train
sumber,,
multiview,844,3194
pushup,168,684
qevd,1215,6632


orang/video yang kepecah ke train & test: 0 (harus 0)


## 7. Model kelas (gerakan apa)

Split pakai kolom `split` dari CSV. Skor yang dilaporkan: window-level dan video-level (voting semua window dalam satu video).

In [12]:
F_ALL = summarize(A_all)
Y_KELAS = META.kelas.map({k: i for i, k in enumerate(KELAS)}).to_numpy()


def make_model():
    return RandomForestClassifier(n_estimators=300, max_depth=12, min_samples_leaf=3,
                                  class_weight="balanced", n_jobs=-1, random_state=42)


def vote(video_ids, y_true, y_pred):
    t = pd.DataFrame({"v": video_ids, "y": y_true, "p": y_pred})
    return t.groupby("v").agg(y=("y", "first"), p=("p", lambda s: s.mode().iloc[0]))


def evaluate(y, pred, vids, names, title):
    v = vote(vids, y, pred)
    print(f"=== {title} ===")
    print(f"window accuracy: {accuracy_score(y, pred):.3f} ({len(y)} window) | "
          f"video accuracy: {accuracy_score(v.y, v.p):.3f} ({len(v)} video)")
    labels = list(range(len(names)))
    print(classification_report(v.y, v.p, labels=labels, target_names=names, zero_division=0))
    print("confusion matrix (video, baris = asli, kolom = prediksi):")
    print(pd.DataFrame(confusion_matrix(v.y, v.p, labels=labels), index=names, columns=names))
    return v


tr, te = (META.split == "train").to_numpy(), (META.split == "test").to_numpy()
m_kelas = make_model().fit(F_ALL[tr], Y_KELAS[tr])
v_kelas = evaluate(Y_KELAS[te], m_kelas.predict(F_ALL[te]), META.video_id[te].to_numpy(), KELAS, "KELAS, train -> test")

# akurasi per sumber (test saja)
for s in SUMBER:
    ms = te & (META.sumber == s).to_numpy()
    if ms.sum():
        vs = vote(META.video_id[ms].to_numpy(), Y_KELAS[ms], m_kelas.predict(F_ALL[ms]))
        print(f"  video accuracy {s:10s}: {accuracy_score(vs.y, vs.p):.3f} ({len(vs)} video test)")

imp = pd.Series(m_kelas.feature_importances_, index=FEAT_COLS).sort_values(ascending=False)
print("\n10 fitur terpenting:"); display(imp.head(10).round(3))

=== KELAS, train -> test ===
window accuracy: 0.850 (2227 window) | video accuracy: 0.805 (266 video)
                 precision    recall  f1-score   support

    Elbow_Plank       0.73      0.96      0.83        23
     High_Plank       0.92      0.98      0.95        56
         Pushup       0.97      0.56      0.71       106
Pushup_on_Knees       0.36      0.91      0.51        22
     Side_Plank       0.98      0.98      0.98        59

       accuracy                           0.80       266
      macro avg       0.79      0.88      0.80       266
   weighted avg       0.89      0.80      0.81       266

confusion matrix (video, baris = asli, kolom = prediksi):
                 Elbow_Plank  High_Plank  Pushup  Pushup_on_Knees  Side_Plank
Elbow_Plank               22           0       0                0           1
High_Plank                 1          55       0                0           0
Pushup                     6           5      59               36           0
Pushup_on_Kn

bahu_kanan_min     0.079
siku_kiri_std      0.053
siku_kiri_range    0.047
bahu_kanan_max     0.046
bahu_kanan_mean    0.044
bahu_kiri_std      0.044
bahu_kiri_min      0.042
siku_kanan_mean    0.040
siku_kanan_min     0.037
bahu_kanan_std     0.034
dtype: float64

## 8. Model kualitas (good / bad)

Cuma pakai window yang punya label good/bad (`multiview` + `pushup`, semuanya `Pushup`). Selain tes train/test biasa ada **tes lintas sumber**: latih di satu sumber, tes di sumber lain. Itu angka yang paling jujur soal apakah model bisa dipakai di rekaman baru.

In [13]:
NAMES_Q = ["bad", "good"]
has_q = META.kualitas.isin(["good", "bad"]).to_numpy()
Y_Q = (META.kualitas == "good").astype(int).to_numpy()
print("window berlabel kualitas:", int(has_q.sum()), "| per kelas:", META[has_q].kelas.value_counts().to_dict())

if has_q.sum() == 0:
    print("tidak ada data berlabel kualitas, bagian ini dilewati")
else:
    m_q = make_model().fit(F_ALL[tr & has_q], Y_Q[tr & has_q])
    evaluate(Y_Q[te & has_q], m_q.predict(F_ALL[te & has_q]), META.video_id[te & has_q].to_numpy(),
             NAMES_Q, "KUALITAS, train -> test")

    for a, b in (("multiview", "pushup"), ("pushup", "multiview")):
        ma, mb = has_q & (META.sumber == a).to_numpy(), has_q & (META.sumber == b).to_numpy()
        if ma.sum() and mb.sum() and len(set(Y_Q[ma])) > 1:
            m = make_model().fit(F_ALL[ma], Y_Q[ma])
            evaluate(Y_Q[mb], m.predict(F_ALL[mb]), META.video_id[mb].to_numpy(),
                     NAMES_Q, f"KUALITAS lintas sumber: latih {a} -> tes {b}")

window berlabel kualitas: 4890 | per kelas: {'Pushup': 4890}
=== KUALITAS, train -> test ===
window accuracy: 0.753 (1012 window) | video accuracy: 0.760 (50 video)
              precision    recall  f1-score   support

         bad       0.76      0.76      0.76        25
        good       0.76      0.76      0.76        25

    accuracy                           0.76        50
   macro avg       0.76      0.76      0.76        50
weighted avg       0.76      0.76      0.76        50

confusion matrix (video, baris = asli, kolom = prediksi):
      bad  good
bad    19     6
good    6    19
=== KUALITAS lintas sumber: latih multiview -> tes pushup ===
window accuracy: 0.700 (852 window) | video accuracy: 0.750 (100 video)
              precision    recall  f1-score   support

         bad       0.73      0.80      0.76        50
        good       0.78      0.70      0.74        50

    accuracy                           0.75       100
   macro avg       0.75      0.75      0.75       

## 9. Model final + simpan

Latih ulang dengan **semua** data (train + test) dan simpan ke `outputs/`. Jangan dievaluasi, modelnya sudah lihat semua video. Angka jujur ada di bagian 7 dan 8.

Dataset disimpan sebagai `dataset_final_v3.npz` + `dataset_windows_v3.csv`. Coding `y_kualitas`: `bad=0, good=1, unknown=-1`.

In [14]:
meta_info = dict(window=WINDOW, stride=STRIDE, kelas=KELAS, sudut=SUDUT_DEF, cols=FEAT_COLS)

final_kelas = make_model().fit(F_ALL, Y_KELAS)
joblib.dump({"model": final_kelas, **meta_info}, OUT_DIR / "kelas_model.pkl")
print("kelas_model.pkl  <-", len(F_ALL), "window,", META.video_id.nunique(), "video")

if has_q.sum():
    final_q = make_model().fit(F_ALL[has_q], Y_Q[has_q])
    joblib.dump({"model": final_q, "names": NAMES_Q, **meta_info}, OUT_DIR / "kualitas_model.pkl")
    print("kualitas_model.pkl <-", int(has_q.sum()), "window,", META[has_q].video_id.nunique(), "video")

y_kual = np.where(META.kualitas == "good", 1, np.where(META.kualitas == "bad", 0, -1))
np.savez_compressed(
    ROOT / "dataset_final_v3.npz",
    X=X_all, A=A_all, y_kelas=Y_KELAS, y_kualitas=y_kual,
    split=META.split.to_numpy(dtype=str), sumber=META.sumber.to_numpy(dtype=str),
    video_id=META.video_id.to_numpy(dtype=str),
    kelas_names=np.array(KELAS), nama_sudut=np.array(NAMA_SUDUT))
META.drop(columns="group").to_csv(ROOT / "dataset_windows_v3.csv", index=False)
print("tersimpan: dataset_final_v3.npz, dataset_windows_v3.csv")

kelas_model.pkl  <- 12737 window, 1668 video
kualitas_model.pkl <- 4890 window, 242 video
tersimpan: dataset_final_v3.npz, dataset_windows_v3.csv


## 10. Coba satu video

Pakai model final. Hasilnya: gerakan apa, dan kalau gerakannya punya model kualitas (sekarang cuma `Pushup`), good atau bad.
Frame sampling otomatis menyesuaikan fps video ke sekitar `TARGET_FPS` (data QEVD dan multiview di CSV efektifnya 15 fps).

In [15]:
def predict_video(path, target_fps=15):
    import cv2
    cap = cv2.VideoCapture(str(path))
    fps = cap.get(cv2.CAP_PROP_FPS) or 30
    cap.release()
    step = max(1, round(fps / target_fps))

    kel, kq = joblib.load(OUT_DIR / "kelas_model.pkl"), None
    if (OUT_DIR / "kualitas_model.pkl").exists():
        kq = joblib.load(OUT_DIR / "kualitas_model.pkl")
    kp = video_to_keypoints(path, step)
    ws = make_windows(kp)
    if not ws:
        print("tidak ada window yang valid (video terlalu pendek atau pose tidak kebaca)")
        return
    F = summarize(angles(np.stack([w for _, w in ws])))
    pk = kel["model"].predict(F)
    kelas = KELAS[int(pd.Series(pk).mode().iloc[0])]
    print(f"{Path(path).name}: {kelas} ({(pk == KELAS.index(kelas)).mean():.0%} dari {len(pk)} window)")
    if kq is not None and kelas == "Pushup":
        pq = kq["model"].predict(F)
        print(f"  form: {'GOOD' if pq.mean() > 0.5 else 'BAD'} ({pq.mean():.0%} window good)")
    elif kelas != "Pushup":
        print("  belum ada model kualitas untuk kelas ini")


# predict_video("contoh.mp4")

In [16]:
# DIAGNOSA 1: video mana yang salah di model kelas
pred = m_kelas.predict(F_ALL[te])
t = pd.DataFrame({"video_id": META.video_id[te].to_numpy(), "asli": Y_KELAS[te], "pred": pred})
v = t.groupby("video_id").agg(asli=("asli", "first"), pred=("pred", lambda s: s.mode().iloc[0])).reset_index()
v["asli"], v["pred"] = v.asli.map(lambda i: KELAS[i]), v.pred.map(lambda i: KELAS[i])
v = v.merge(idx[["id", "sumber", "file"]], left_on="video_id", right_on="id").drop(columns="id")
salah = v[v.asli != v.pred]
print(f"video salah: {len(salah)} dari {len(v)}")
display(salah.groupby(["sumber", "asli", "pred"]).size().rename("jumlah").reset_index()
        .sort_values("jumlah", ascending=False).head(10))
print("\ncontoh video Pushup yang ditebak Pushup_on_Knees (tonton beberapa untuk dicek):")
display(salah[(salah.asli == "Pushup") & (salah.pred == "Pushup_on_Knees")][["sumber", "file"]].head(15))

video salah: 52 dari 266


,sumber,asli,pred,jumlah
5,qevd,Pushup,Pushup_on_Knees,36
3,qevd,Pushup,Elbow_Plank,5
4,qevd,Pushup,High_Plank,5
0,pushup,Pushup,Elbow_Plank,1
2,qevd,High_Plank,Elbow_Plank,1
1,qevd,Elbow_Plank,Side_Plank,1
6,qevd,Pushup_on_Knees,Elbow_Plank,1
7,qevd,Pushup_on_Knees,Pushup,1
8,qevd,Side_Plank,Pushup,1



contoh video Pushup yang ditebak Pushup_on_Knees (tonton beberapa untuk dicek):


,sumber,file
133,qevd,00281820.mp4
152,qevd,00284618.mp4
155,qevd,00284993.mp4
156,qevd,00285141.mp4
157,qevd,00285339.mp4
160,qevd,00285642.mp4
162,qevd,00285991.mp4
163,qevd,00286016.mp4
165,qevd,00286394.mp4
170,qevd,00286872.mp4
